# True-Only Baseline Training and Evaluation

This notebook trains the `true_only` baseline, which uses only the clean normalized state slice and does **not** participate in noise sweeps.

## Output layout

This notebook saves outputs to the new project structure:

- Checkpoints: `artifacts/checkpoints/true_only/<env>/seed_<seed>/`
- Metrics: `results/raw_metrics/true_only/<env>/seed_<seed>/`
- Observation stats: `artifacts/obs_stats/true_only/<env>/seed_<seed>/`

## Recommended usage

Run this notebook once per environment and per seed. Unlike noisy methods, it should **not** expand over noise dimension, noise scale, or noise type.


In [ ]:
# Cell 1: project setup and imports
from pathlib import Path
import json
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate the project root containing the 'src' directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import NoisyOfflineRLDataset
from src.iql import IQLAgent
from src.train_eval import eval_policy_on_env


In [ ]:
# Cell 2: all tunable parameters
ENV_NAME = "halfcheetah-medium-v2"
SEED = 1

BATCH_SIZE = 256
EPOCHS = 100
SAVE_EVERY = 10

EXPECTILE = 0.7
TEMPERATURE = 3.0
DISCOUNT = 0.99

EVAL_EPISODES = 20
EVAL_MAX_STEPS = 1000
USE_TIMEOUTS = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
METHOD = "true_only"


In [ ]:
# Cell 3: output paths for the true_only baseline
CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / METHOD
    / ENV_NAME
    / f"seed_{SEED}"
)
METRICS_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw_metrics"
    / METHOD
    / ENV_NAME
    / f"seed_{SEED}"
)
OBS_STATS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "obs_stats"
    / METHOD
    / ENV_NAME
    / f"seed_{SEED}"
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_STATS_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Metrics directory:", METRICS_DIR)
print("Obs-stats directory:", OBS_STATS_DIR)


In [ ]:
# Cell 4: build the dataset, data loader, and observation statistics
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=0,
    noise_scale=0.0,
    seed=SEED,
    use_timeouts=USE_TIMEOUTS,
    noise_type="concat",
)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim

print("Noisy state dimension:", state_dim)
print("Clean state dimension:", true_state_dim)
print("Action dimension:", action_dim)

stats_path = OBS_STATS_DIR / "obs_stats.npz"
np.savez(
    stats_path,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)
print("Saved observation statistics to:", stats_path)


In [ ]:
# Cell 5: define the true_only training function
def train_true_only_baseline(
    train_loader,
    action_dim,
    true_state_dim,
    device,
    epochs,
    expectile,
    temperature,
    discount,
    checkpoint_dir,
    save_every=10,
):
    """Train IQL using only the clean state slice returned by the dataset."""
    iql = IQLAgent(
        latent_dim=true_state_dim,
        action_dim=action_dim,
        device=device,
        expectile=expectile,
        temperature=temperature,
        discount=discount,
    )

    history = []

    for epoch in range(1, epochs + 1):
        v_losses, q_losses, a_losses = [], [], []

        for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in train_loader:
            act = act.to(device)
            rew = rew.to(device)
            done = done.to(device)
            pure_obs = pure_obs.to(device)
            pure_next_obs = pure_next_obs.to(device)

            z = pure_obs
            next_z = pure_next_obs

            v_loss, q_loss, a_loss = iql.train_step(z, act, next_z, rew, done)
            v_losses.append(v_loss)
            q_losses.append(q_loss)
            a_losses.append(a_loss)

        avg_v = float(np.mean(v_losses))
        avg_q = float(np.mean(q_losses))
        avg_a = float(np.mean(a_losses))

        history.append(
            {
                "epoch": epoch,
                "value_loss": avg_v,
                "q_loss": avg_q,
                "actor_loss": avg_a,
            }
        )

        print(f"[IQL][true_only] epoch {epoch}: V={avg_v:.4f}, Q={avg_q:.4f}, A={avg_a:.4f}")

        if epoch % save_every == 0 or epoch == epochs:
            ckpt_path = checkpoint_dir / f"iql_true_only_epoch_{epoch}.pth"
            torch.save(
                {
                    "actor": iql.actor.state_dict(),
                    "q_net": iql.q_net.state_dict(),
                    "v_net": iql.v_net.state_dict(),
                    "meta": {
                        "method": "true_only",
                        "env_name": ENV_NAME,
                        "seed": SEED,
                        "epoch": epoch,
                    },
                },
                ckpt_path,
            )
            print("Saved checkpoint:", ckpt_path)

    return iql, history


In [ ]:
# Cell 6: run training
iql, history = train_true_only_baseline(
    train_loader=train_loader,
    action_dim=action_dim,
    true_state_dim=true_state_dim,
    device=DEVICE,
    epochs=EPOCHS,
    expectile=EXPECTILE,
    temperature=TEMPERATURE,
    discount=DISCOUNT,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
)


In [ ]:
# Cell 7: evaluate the trained policy and save metrics
print("Start evaluating...")

metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=None,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=0,
    noise_scale=0.0,
    noise_type="concat",
    episodes=EVAL_EPISODES,
    max_steps=EVAL_MAX_STEPS,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = METRICS_DIR / "metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "env_name": ENV_NAME,
            "method": METHOD,
            "seed": SEED,
            "training_history": history,
            **metrics,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved metrics:", metrics_path)
print(metrics)
